# 01 - Setup PostGIS Extensions on Lakebase

Connects to the Lakebase Postgres instance and installs the PostGIS and
hstore extensions required by Nominatim. Also creates the `www-data`
role that Nominatim expects.

Authenticates via `WorkspaceClient` (notebook runs in the same workspace).

## Configuration

In [ ]:
%pip install databricks-sdk==0.99.0
%restart_python

In [ ]:
dbutils.widgets.text("pg_project_id", "nominatim-lakebase", "Lakebase Project ID")
dbutils.widgets.text("pg_branch_id", "production", "Lakebase Branch ID")
dbutils.widgets.text("pg_user", "justin.monaldo@databricks.com", "PG User")
dbutils.widgets.text("pg_database", "nominatim", "PG Database")
dbutils.widgets.text("pg_port", "5432", "PG Port")

In [ ]:
pg_project_id = dbutils.widgets.get("pg_project_id")
pg_branch_id = dbutils.widgets.get("pg_branch_id")
pg_user = dbutils.widgets.get("pg_user")
pg_database = dbutils.widgets.get("pg_database")
pg_port = dbutils.widgets.get("pg_port")

In [ ]:
%run ./_helpers

In [ ]:
import psycopg2
from psycopg2 import sql

pg_env = get_pg_env(
    project_id=pg_project_id,
    branch_id=pg_branch_id,
    user=pg_user,
    database=pg_database,
    port=pg_port,
)
conn_params = get_psycopg_conn_params(pg_env)
pg_host = pg_env["PGHOST"]

## Check / Create Target Database

In [ ]:
print("=" * 60)
print("PostGIS Setup for Nominatim")
print("=" * 60)
print(f"\n  Host:     {pg_host}")
print(f"  Database: {pg_database}")
print(f"  User:     {pg_user}")
print()

# Connect to maintenance DB first
maint_params = {**conn_params, "dbname": "postgres"}
conn = psycopg2.connect(**maint_params)
conn.autocommit = True
cur = conn.cursor()

# Check PostgreSQL version
cur.execute("SELECT version();")
version = cur.fetchone()[0]
print(f"PostgreSQL: {version}\n")

# Check if target database exists; create if not
cur.execute("SELECT 1 FROM pg_database WHERE datname = %s;", (pg_database,))
if cur.fetchone():
    print(f"Database '{pg_database}' already exists")
else:
    print(f"Creating database '{pg_database}'...")
    cur.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(pg_database)))
    print(f"Database '{pg_database}' created successfully")

cur.close()
conn.close()

## Install Extensions

In [ ]:
conn = psycopg2.connect(**conn_params)
conn.autocommit = True
cur = conn.cursor()

# PostGIS
print("Installing PostGIS extension...")
cur.execute("CREATE EXTENSION IF NOT EXISTS postgis;")
print("  PostGIS extension created successfully")

# hstore
print("Installing hstore extension...")
cur.execute("CREATE EXTENSION IF NOT EXISTS hstore;")
print("  hstore extension created successfully")

## Verify Extensions

In [ ]:
cur.execute("""
    SELECT extname, extversion
    FROM pg_extension
    WHERE extname IN ('postgis', 'hstore')
    ORDER BY extname;
""")
extensions = cur.fetchall()

print("Installed extensions:")
for ext_name, ext_version in extensions:
    print(f"  - {ext_name}: version {ext_version}")

cur.execute("SELECT PostGIS_Version();")
postgis_version = cur.fetchone()[0]
print(f"\nPostGIS version: {postgis_version}")

## Create www-data User

In [ ]:
cur.execute("SELECT 1 FROM pg_roles WHERE rolname = 'www-data';")
if cur.fetchone():
    print("www-data user already exists")
else:
    cur.execute('CREATE USER "www-data";')
    print("www-data user created successfully")

cur.close()
conn.close()

print("\n" + "=" * 60)
print("PostGIS setup completed successfully!")
print("=" * 60)
print("\nNext: Run 02_download_osm_data to download OSM .pbf files")